In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
model_name = os.getenv("LOCAL_MODE")
base_url = os.getenv("LOCAL_BASE_URL")

In [8]:
# from langchain_ollama import OllamaLLM
# llm = OllamaLLM(
#     model=model_name,
#     base_url=base_url,
#     temperature=0.2
# )

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                           # 超时保护（秒）
    max_retries=2                           # 简单重试
)

In [9]:
from pydantic import BaseModel,Field

# 1. 使用 Pydantic v2 语法严格定义期望的输出 Schema
class StructuredResponse(BaseModel):
    """定义一个结构化的响应模型"""
    answer: str = Field(description="对用户问题的直接回答")
    followup_question: str = Field(description="用于深入挖掘用户需求的后续问题")
    confidence_score: float = Field(description="回答的置信度，范围0-1")

# 2. 关键修复：显式指定 method 参数以适配 Pydantic v2 模型
structured_model = llm.with_structured_output(
    StructuredResponse,
    method="function_calling"  # 显式指定使用函数调用方法，这是处理 Pydantic v2 模型的推荐方式
)

# 3. 调用模型并获取结构化输出对象
structured_output = structured_model.invoke("爱因斯坦最著名的成就是什么？")

# 4. 像操作普通的 Python 对象一样直接访问各个字段
print(f"答案：{structured_output.answer}")
print(f"置信度：{structured_output.confidence_score}")
print(f"后续问题：{structured_output.followup_question}")

# 5. Pydantic v2 中，使用 .model_dump() 来转换为字典
result_dict = structured_output.model_dump()
print(f"转换为字典：{result_dict}")

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}